# Creating synthetic log for a two-role, two-specializations setting

There are only two specializations in a process, according to an attribute on the case level, e.g., customer type being either “General” or “VIP”. There exists a total of 20 resources assigned to two roles, Role-1 and Role-2, with 10 resources per role. There are two process tasks (activities), namely A and B. Any of the 10 resources in Role-1 can perform both task A and task B, but only for “General” cases; while the 10 resources in Role-2 can perform A and B only for “VIP” cases.  

In [1]:
from os.path import join as path_join
from datetime import datetime, timedelta
from itertools import cycle

import pandas as pd
import altair as alt

## Generate the log

In [2]:
NUM_CASES = 1000

log = []
# 1. Generate cases, each with 2 events, for activities A and B, respectively
for i in range(NUM_CASES):
    log.append({
        'case_id': i + 1,
        'activity': 'A',
    })
    log.append({
        'case_id': i + 1,
        'activity': 'B',
    })
log = pd.DataFrame(log)

# 2. Add case attribute "customer_type"
# 50% cases are "General"; 50% cases are "VIP"
log.loc[
    log['case_id'].isin(range(1, int(0.5 * NUM_CASES) + 1)), 
    'case_customer_type'
] = 'General'
log.loc[
    log['case_id'].isin(range(int(0.5 * NUM_CASES) + 1, NUM_CASES + 1)), 
    'case_customer_type'
] = 'VIP'

# 3. Add resources
# NOTE: Two roles "Role_1", "Role_2"
#       Each with 10 resources, with IDs in format "Role_1-Resource_X"
# NOTE: Role_1 resources can only perform for customer_type = "General"
#       Role_2 resources can only perform for customer_type = "VIP"
#       But all resources can perform both activities A and B 
for customer_type, role in {'General': 'Role_1', 'VIP': 'Role_2'}.items():
    resources = [f'{role}-Resource_{j+1}' for j in range(10)]
    # cycle the assignment to ensure all resources will perform both A and B
    pool = cycle(sorted(resources) + sorted(resources, reverse=True))
    for i, event in log[log['case_customer_type'] == customer_type].iterrows():
        log.loc[i, 'resource'] = next(pool)

# 4. Add a dummy timestamp, assuming all activities happened on the same date
# NOTE: This ensures that resources are NOT distinct by the datetime when they
# executed the activities.
log['timestamp'] = datetime(2026, 1, 1, 0, 0, 0)

log

,case_id,activity,case_customer_type,resource,timestamp
0,1,A,General,Role_1-Resource_1,2026-01-01
1,1,B,General,Role_1-Resource_10,2026-01-01
2,2,A,General,Role_1-Resource_2,2026-01-01
3,2,B,General,Role_1-Resource_3,2026-01-01
4,3,A,General,Role_1-Resource_4,2026-01-01
...,...,...,...,...,...
1995,998,B,VIP,Role_2-Resource_4,2026-01-01
1996,999,A,VIP,Role_2-Resource_3,2026-01-01
1997,999,B,VIP,Role_2-Resource_2,2026-01-01
1998,1000,A,VIP,Role_2-Resource_10,2026-01-01


In [3]:
# Rename dataframe columns per XES convention
log = log.rename(columns={
    'case_id': 'case:concept:name',
    'activity': 'concept:name',
    'resource': 'org:resource',
    'timestamp': 'time:timestamp',
    'case_customer_type': 'case:customer_type',
})

In [4]:
# Print key stats

NUM_EVENTS = len(log)
print('Number of events: {}'.format(NUM_EVENTS))
NUM_CASES = log['case:concept:name'].nunique()
print('Unique cases by case ID (case:concept:name): {}'.format(NUM_CASES))
print('Unique activity names (concept:name): {}'.format(log['concept:name'].unique()))

print('Unique number of resources by resource ID (org:resource): {}'.format(
    log['org:resource'].nunique()
))
# Parse the role information from resource IDs
log['org:role'] = log['org:resource'].apply(lambda s: s.split('-')[0])
print('Unique resource roles: {}'.format(log['org:role'].unique()))
print('Unique resources per each role:')
for role in log['org:role'].unique():
    print(sorted(log.loc[log['org:role'] == role, 'org:resource'].unique()))

print('-' * 79)
print('Unique values of case attribute "case:customer_type": {}'.format(
    log['case:customer_type'].unique())
)


Number of events: 2000
Unique cases by case ID (case:concept:name): 1000
Unique activity names (concept:name): ['A' 'B']
Unique number of resources by resource ID (org:resource): 20
Unique resource roles: ['Role_1' 'Role_2']
Unique resources per each role:
['Role_1-Resource_1', 'Role_1-Resource_10', 'Role_1-Resource_2', 'Role_1-Resource_3', 'Role_1-Resource_4', 'Role_1-Resource_5', 'Role_1-Resource_6', 'Role_1-Resource_7', 'Role_1-Resource_8', 'Role_1-Resource_9']
['Role_2-Resource_1', 'Role_2-Resource_10', 'Role_2-Resource_2', 'Role_2-Resource_3', 'Role_2-Resource_4', 'Role_2-Resource_5', 'Role_2-Resource_6', 'Role_2-Resource_7', 'Role_2-Resource_8', 'Role_2-Resource_9']
-------------------------------------------------------------------------------
Unique values of case attribute "case:customer_type": ['General' 'VIP']


In [5]:
# Verify task assignment

# NOTE: Expecting that any resource role can perform any task 
#       In total 4 unique combinations, e.g., (Role_1, A)
print('Unique pairs of role - activity name: {}'.format(
    log[['org:role', 'concept:name']].drop_duplicates())
)

# NOTE: Expecting that Role_1 resources can perform for "General" cases
#       Role_2 resources can perform for "VIP" cases
#       In total 2 unique combinations
print('Unique pairs of role - customer_type: {}'.format(
    log[['org:role', 'case:customer_type']].drop_duplicates())
)

Unique pairs of role - activity name:      org:role concept:name
0      Role_1            A
1      Role_1            B
1000   Role_2            A
1001   Role_2            B
Unique pairs of role - customer_type:      org:role case:customer_type
0      Role_1            General
1000   Role_2                VIP


## Visualize the event distribution for validation

In [6]:
# Visualize workload distribution to check 
# if resources within the same role are fully interchangeable,
# and if resources with different roles are distinct

# resource - activity
data = log.groupby([
    'org:resource', 'concept:name'
]).size().reset_index().rename(columns={
    'org:resource': 'resource',
    'concept:name': 'activity',
    0: 'count'
})
alt.Chart(
    data=data,
).mark_rect().encode(
    x=alt.X('resource:N', title='org:resource (resource)'), 
    y=alt.Y('activity:N', title='concept:name (activity)'),
    color=alt.Color('count:Q', title='Number of events')
)

alt.Chart(...)

In [7]:
# resource - case (customer_type)
data = log.drop_duplicates(subset=['org:resource', 'case:concept:name']).groupby([
    'org:resource', 'case:customer_type'
]).size().reset_index().rename(columns={
    'org:resource': 'resource',
    'case:customer_type': 'customer_type',
    0: 'count'
})
alt.Chart(
    data=data
).mark_rect().encode(
    x=alt.X('resource:N', title='org:resource (resource)'), 
    y=alt.Y('customer_type:N', title='case:customer_type'),
    color=alt.Color('count:Q', title='Number of case IDs').scale(domain=[0, NUM_CASES])
)

alt.Chart(...)

## Export log

In [8]:
log.to_csv(path_join('./data', 'synthetic.csv'), index=False)